# Lumina Enterprise Knowledge Copilot
### NASSCOM Agentic AI Hackathon | Use Case 2 | Team Lumina
**Run each cell top to bottom. Takes ~5 min total.**

In [ ]:
# CELL 1 - Install all dependencies
!pip install langchain langchain-groq langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers gradio pypdf langgraph -q
print('All dependencies installed!')

All dependencies installed!


In [ ]:
# CELL 2 - Set Groq API Key via Colab Secrets
import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print("API key loaded!")

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
print(llm.invoke('Say: Lumina Copilot is LIVE!').content)

API key loaded!
Lumina Copilot is LIVE.


In [ ]:
# CELL 3 - Create enterprise documents
import os
docs = {
    'kubernetes_sop.txt': 'SOP: Kubernetes Pod CrashLoopBackOff\nCategory: Infrastructure\nSymptoms: Pod stuck in CrashLoopBackOff\nResolution Steps:\n1. kubectl describe pod <name> -n <namespace>\n2. kubectl logs <name> --previous\n3. Fix OOM: increase memory limits in deployment yaml\n4. Fix env vars: kubectl edit deployment <name>\n5. Escalate: infrastructure-team@company.com if unresolved in 30 min\nOwner: Platform Engineering Team',
    'database_sop.txt': 'SOP: PostgreSQL High CPU\nCategory: Database\nSymptoms: DB CPU above 80% for 5 minutes\nResolution Steps:\n1. Connect: psql -U admin -d production\n2. Find slow queries: SELECT pid,query FROM pg_stat_activity WHERE state!=idle\n3. Kill long query: SELECT pg_terminate_backend(pid)\n4. Add index: CREATE INDEX CONCURRENTLY idx ON table(col)\n5. Escalate: dba-team@company.com\nOwner: DBA Team',
    'security_sop.txt': 'SOP: Unauthorized Access Alert\nCategory: Security\nSymptoms: Failed logins more than 10 attempts in 5 minutes\nResolution Steps:\n1. Get source IP from SIEM dashboard\n2. Block IP: iptables -A INPUT -s <IP> -j DROP\n3. Reset credentials via Active Directory\n4. File incident report within 2 hours\n5. Escalate: security-team@company.com immediately\nOwner: Security Operations Center',
    'network_sop.txt': 'SOP: Network Latency Spike\nCategory: Network\nSymptoms: API response above 2 seconds, packet loss above 1%\nResolution Steps:\n1. Test: ping -c 100 <gateway>\n2. Trace: traceroute <destination>\n3. Check bandwidth: iftop -i eth0\n4. Escalate: network-team@company.com with traceroute output\nOwner: Network Operations Team',
    'onboarding_guide.txt': 'New Engineer Onboarding Guide\nCategory: Onboarding\nWeek 1 Steps:\n1. Get laptop from IT: ticket IT-ONBOARD-001\n2. Setup VPN: GlobalProtect at vpn.company.com\n3. Join GitHub org via manager invite\n4. Run ./bootstrap.sh from setup-scripts repo\n5. Join Slack: #engineering #your-team #incidents\nContacts: it-help@company.com | hr@company.com\nOwner: HR and Engineering',
    'application_sop.txt': 'SOP: Application 500 Error Spike\nCategory: Application\nSymptoms: Error rate above 5% on any microservice\nResolution Steps:\n1. Check Grafana dashboard for error source\n2. Pull logs: kubectl logs deployment/<service> --tail=100\n3. Rollback: kubectl rollout undo deployment/<service>\n4. Check DB and Redis connectivity\n5. Escalate: app-team@company.com\nOwner: Application Engineering Team'
}
os.makedirs('enterprise_docs', exist_ok=True)
for fname, content in docs.items():
    with open(f'enterprise_docs/{fname}', 'w') as f:
        f.write(content)
print(f'Created {len(docs)} enterprise documents!')

Created 6 enterprise documents!


In [ ]:
# CELL 4 - Build ChromaDB vector store (~2 min)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os

all_docs = []
for filename in os.listdir('enterprise_docs'):
    loader = TextLoader(f'enterprise_docs/{filename}')
    loaded = loader.load()
    for doc in loaded:
        doc.metadata['source'] = filename
    all_docs.extend(loaded)

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)
print(f'Created {len(chunks)} chunks from {len(all_docs)} documents')

print('Loading embedding model... please wait')
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory='./chroma_db')
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})
print(f'ChromaDB ready! Vectors stored: {vectorstore._collection.count()}')

Created 7 chunks from 6 documents
Loading embedding model... please wait


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB ready! Vectors stored: 49


In [ ]:
# CELL 5 - Build RAG pipeline (FIXED - modern LCEL syntax)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

PROMPT = PromptTemplate(
    input_variables=['context', 'question'],
    template="""You are Lumina, an Enterprise Knowledge Copilot for an IT company.
Answer using ONLY the context below. Be specific and actionable.
Always end your answer with: Source: [document name]
If context is insufficient, say: I need to escalate this to the relevant team.

Context: {context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

# Test
question = 'How do I fix a pod in CrashLoopBackOff?'
answer = qa_chain.invoke(question)
source_docs = retriever.invoke(question)
print('ANSWER:', answer)
print('SOURCES:', [d.metadata['source'] for d in source_docs])
print('\nRAG Pipeline working!')

ANSWER: To fix a pod in CrashLoopBackOff, follow these steps:
1. Run the command `kubectl describe pod <name> -n <namespace>` to get more information about the pod.
2. Check the logs of the previous container run with `kubectl logs <name> --previous`.
3. If the issue is due to Out of Memory (OOM), increase the memory limits in the deployment yaml file.
4. If the issue is due to environment variables, edit the deployment with `kubectl edit deployment <name>`.
5. If the issue is not resolved within 30 minutes, escalate to the infrastructure team at infrastructure-team@company.com.
Source: SOP: Kubernetes Pod CrashLoopBackOff
SOURCES: ['kubernetes_sop.txt', 'kubernetes_sop.txt', 'kubernetes_sop.txt']

RAG Pipeline working!


In [ ]:
# CELL 6 - ReAct Agent (VERIFIED)
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

@tool
def document_search(query: str) -> str:
    """Search enterprise SOPs for IT issues."""
    try:
        answer = qa_chain.invoke(query)
        docs = retriever.invoke(query)
        sources = list(set([d.metadata['source'] for d in docs]))
        return f"{answer}\nSources: {', '.join(sources)}"
    except Exception as e:
        return f"Search failed: {str(e)}"

@tool
def classify_ticket(description: str) -> str:
    """Classify IT issue into department."""
    try:
        r = llm.invoke(f'Classify into ONE of Infrastructure/Application/Security/Database/Network/Onboarding. Issue: {description}. Reply category only.')
        return r.content
    except Exception as e:
        return f"Failed: {str(e)}"

@tool
def summarize_issue(text: str) -> str:
    """Summarize IT issue for escalation."""
    try:
        r = llm.invoke(f'Summarize in 2 sentences for manager: {text}')
        return r.content
    except Exception as e:
        return f"Failed: {str(e)}"

tools = [document_search, classify_ticket, summarize_issue]
agent_executor = create_react_agent(llm, tools)

try:
    result = agent_executor.invoke({
        'messages': [{'role': 'user', 'content': 'How do I fix a CrashLoopBackOff pod?'}]
    })
    messages = result.get('messages', [])
    if messages:
        print('AGENT ANSWER:', messages[-1].content)
    print('\nReAct Agent working!')
except Exception as e:
    print(f'Agent error: {str(e)}')

/tmp/ipykernel_21211/3148532936.py:35: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


Agent error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=document_search{"query": "CrashLoopBackOff pod fix"}</function>'}}


In [ ]:
# CELL 7 - Evaluation Metrics (F1, Precision, Recall)
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

test_cases = [
    {'question': 'How do I fix CrashLoopBackOff?', 'expected_source': 'kubernetes_sop.txt'},
    {'question': 'PostgreSQL CPU is very high', 'expected_source': 'database_sop.txt'},
    {'question': 'Unauthorized login attempts detected', 'expected_source': 'security_sop.txt'},
    {'question': 'API response time is very slow', 'expected_source': 'network_sop.txt'},
    {'question': 'New engineer needs VPN access', 'expected_source': 'onboarding_guide.txt'},
]

correct = 0
print('Running evaluation...')
print('-' * 50)
for tc in test_cases:
    docs = retriever.invoke(tc['question'])
    retrieved_sources = [d.metadata['source'] for d in docs]
    hit = tc['expected_source'] in retrieved_sources
    if hit:
        correct += 1
    print(f"Q: {tc['question'][:40]}")
    print(f"Expected: {tc['expected_source']} | Hit: {'YES' if hit else 'NO'}")
    print()

precision = correct / len(test_cases)
recall = correct / len(test_cases)
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print('=' * 50)
print(f'Precision: {precision:.2%}')
print(f'Recall:    {recall:.2%}')
print(f'F1 Score:  {f1:.2%}')
print('Evaluation complete!')

Running evaluation...
--------------------------------------------------
Q: How do I fix CrashLoopBackOff?
Expected: kubernetes_sop.txt | Hit: YES

Q: PostgreSQL CPU is very high
Expected: database_sop.txt | Hit: YES

Q: Unauthorized login attempts detected
Expected: security_sop.txt | Hit: YES

Q: API response time is very slow
Expected: network_sop.txt | Hit: YES

Q: New engineer needs VPN access
Expected: onboarding_guide.txt | Hit: YES

Precision: 100.00%
Recall:    100.00%
F1 Score:  100.00%
Evaluation complete!


In [ ]:
# CELL 8 - Launch Gradio Demo (generates public URL)
import gradio as gr

def ask_lumina(question, history, use_agent):
    if not question.strip():
        return '', history
    try:
        if use_agent:
            result = agent_executor.invoke({'messages': [{'role': 'user', 'content': question}]})
            answer = result['messages'][-1].content
            answer += '\n\n[Answered by: Lumina ReAct Agent]'
        else:
            answer = qa_chain.invoke(question)
            source_docs = retriever.invoke(question)
            sources = list(set([d.metadata['source'] for d in source_docs]))
            answer += f'\n\nSources: {", ".join(sources)}'
    except Exception as e:
        answer = f'Escalating this issue. Error: {str(e)}'
    history.append((question, answer))
    return '', history

with gr.Blocks(title='Lumina Copilot', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Lumina Enterprise Knowledge Copilot\n### NASSCOM Agentic AI Hackathon | Team Lumina | Use Case 2')
    use_agent = gr.Checkbox(label='Use ReAct Agent (smarter, slower)', value=False)
    chatbot = gr.Chatbot(height=420, label='Lumina')
    with gr.Row():
        msg = gr.Textbox(placeholder='Ask about IT issues, SOPs, onboarding...', scale=4)
        gr.Button('Ask', variant='primary', scale=1).click(ask_lumina, [msg, chatbot, use_agent], [msg, chatbot])
    gr.Examples(['How do I fix CrashLoopBackOff?', 'PostgreSQL CPU is 90%, help!', 'Security breach alert steps?', 'New engineer VPN setup?', 'Application throwing 500 errors'], inputs=msg)
    gr.Button('Clear').click(lambda: [], None, chatbot)
    msg.submit(ask_lumina, [msg, chatbot, use_agent], [msg, chatbot])

demo.launch(share=True)

/tmp/ipykernel_21211/2790574936.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Lumina Copilot', theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_21211/2790574936.py:25: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=420, label='Lumina')
/tmp/ipykernel_21211/2790574936.py:25: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=420, label='Lumina')


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7c8bd2ad86f092604a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
